# NB-14 · NPV Layer — Cost-Benefit das 25 Prescrições

| Campo | Detalhe |
|---|---|
| **Notebook** | NB-14 — NPV Layer (Camada Operacional do D3) |
| **Cenário** | BYD Camaçari — Polo Automotivo BA, horizonte 2025-2027 |
| **Autor** | Matheus Mendes |
| **Data** | 2026-07-27 |
| **Versão** | 1.0 |
| **Dependências** | `numpy 2.5`, `pandas 3.0`, `matplotlib 3.11`, `plotly 6.9`, `jupyter` |

---

## Sumário

1. **Parâmetros de desconto** — taxa nominal 13% a.a. (NTN-B 10y ~11% + 200bps spread soberano), horizonte 3 anos
2. **Catálogo das 25 ações** — registro operacional completo (AG-001..AG-026, AG-017 REMOVIDA)
3. **Benefício esperado + NPV + ROI por ação** — `benefit = var_exp × cov × p_event`
4. **Counterfactual** — com framework vs sem framework (VaR 4-shock R$ 8,21 bi / CVaR R$ 10,14 bi)
5. **ROI headline** — R$ 3M investido, R$ 200M+/ano evitado, ROI 200× (3 anos), payback < 1 mês
6. **NPV Tornado** — sensibilidade do NPV do portfólio aos drivers globais
7. **Ranking por NPV** — top 1 = AG-022 (CATL LP) com R$ 855M
8. **Decision matrix** — EXECUTAR JÁ (10), EXECUTAR (10), CONDICIONAL (4), MONITORAR (1), REJEITAR (AG-017)

---

**Resultados-chave (cliff notes):**
- 25 ações analisadas (AG-017 catalog-wide REMOVIDA — ROI -67%, contraprova de ação ruim)
- NPV portfólio total: **R$ 3,510 M** em 3 anos a 13%
- Capex total: R$ 1.000 M | custo 3y total: R$ 1.466 M
- Ações com NPV positivo: 24/25 (apenas AG-016 tier system fica marginalmente negativa)
- **Framework D3 sozinho**: investimento R$ 3 M (F1-F3, 28 semanas) → stress evitado R$ 200 M+/ano → **ROI 200× em 3 anos, payback 0,18 mês**
- Top-1: AG-022 (CATL LP 70% locked) — NPV R$ 855 M, ROI 178%
- Tornado dominado por **probabilidade de shock (p)** e **exposição VaR** — não por taxa de desconto
- Sem framework: VaR 4-shock R$ 8,21 bi | CVaR R$ 10,14 bi (resposta manual 4-8h/signal vs 14d detecção + 9,3d ação)

## CELL 1 · Parâmetros de desconto

Taxa nominal de 13% reflete NTN-B 10y (~11%) + 200 bps de spread soberano Brasil. Horizonte de 3 anos alinhado com o ciclo de ramp-up 2025-2027 da planta de Camaçari. Fator anuidade Σ 1/(1+r)^t = 2,3612 é o multiplicador padrão para fluxos uniformes em 3 anos.

In [1]:
import json, warnings, sys, os
warnings.filterwarnings('ignore')
os.environ['MPLBACKEND'] = 'Agg'

NOTEBOOK_ROOT = r'C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva'
OUTPUT_DIR = os.path.join(NOTEBOOK_ROOT, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go

R = 0.13
H = 3
ANNUITY = sum(1 / (1 + R) ** t for t in range(1, H + 1))
print(f'r={R:.0%} | horizonte={H}a | fator anuidade={ANNUITY:.4f}')

def npv(capex, opex, benefit, r=R, h=H):
    return -capex + sum((benefit - opex) / (1 + r) ** t for t in range(1, h + 1))

def cost_pv(capex, opex, r=R, h=H):
    return capex + sum(opex / (1 + r) ** t for t in range(1, h + 1))

print('Funções npv() e cost_pv() prontas')

r=13% | horizonte=3a | fator anuidade=2.3612
Funções npv() e cost_pv() prontas


## CELL 2 · Catálogo das 25 ações + custos

Cada ação tem primitivos auditáveis: `var_exp` (exposição R$ M), `cov` (cobertura esperada 0-1), `p` (probabilidade anual do evento), `capex` (one-shot R$ M), `opex` (recorrente anual R$ M). AG-017 (*Defensivo catalog-wide*) foi **removida** com ROI -67% — ela é a contraprova empírica de que ações amplas sem segmentação destroem valor.

In [2]:
ACTIONS = [
    dict(ag='AG-001', name='Resolver lista suja S7 (kill switch)', dim='S7', prio='CRÍTICA',
         capex=25.0, opex=0.0, var_exp=800, cov=0.60, p=0.30, cond=False, status='EM CURSO'),
    dict(ag='AG-002', name='Implementar hedge FX 95%', dim='S1', prio='CRÍTICA',
         capex=0.0, opex=48.6, var_exp=2100, cov=0.47, p=0.12, cond=False, status='PENDENTE'),
    dict(ag='AG-003', name='Acelerar nacionalização 70%', dim='S8', prio='CRÍTICA',
         capex=150.0, opex=10.0, var_exp=650, cov=0.85, p=0.40, cond=False, status='EM CURSO'),
    dict(ag='AG-004', name='Diferenciação (tier + game theory)', dim='S4/S11', prio='CRÍTICA',
         capex=6.5, opex=0.0, var_exp=500, cov=0.40, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-005', name='Reserva lítio + safety stock', dim='S2', prio='CRÍTICA',
         capex=30.0, opex=0.0, var_exp=400, cov=0.25, p=0.40, cond=False, status='EM CURSO'),
    dict(ag='AG-006', name='Renegociar tariff quota Camex', dim='S10', prio='CRÍTICA',
         capex=24.0, opex=0.0, var_exp=2380, cov=0.35, p=0.35, cond=False, status='PENDENTE'),
    dict(ag='AG-007', name='Definir resposta competitiva (Nash)', dim='S11', prio='CRÍTICA',
         capex=6.0, opex=0.0, var_exp=300, cov=0.30, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-008', name='Plano contingência overcapacity', dim='S11/S9', prio='CRÍTICA',
         capex=4.0, opex=0.0, var_exp=200, cov=0.30, p=0.20, cond=True, status='PENDENTE'),
    dict(ag='AG-009', name='Setup compliance ESG + auditoria', dim='S7', prio='CRÍTICA',
         capex=0.0, opex=0.0, var_exp=200, cov=0.40, p=0.30, cond=False, status='PLANEJADO',
         note='custo consolidado em AG-001'),
    dict(ag='AG-010', name='Monitor T-MV1 (4-shock stress)', dim='S6', prio='CRÍTICA',
         capex=1.4, opex=0.8, var_exp=12800, cov=0.10, p=0.02, cond=False, status='PENDENTE'),
    dict(ag='AG-011', name='Calcular h* baseline (constraint)', dim='S1', prio='ALTA',
         capex=0.05, opex=0.0, var_exp=2100, cov=0.03, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-012', name='Constraint VaR ≤ 20% da margem', dim='S1', prio='ALTA',
         capex=0.05, opex=0.0, var_exp=1500, cov=0.05, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-013', name='Contratar 4 counterparties hedge', dim='S1', prio='ALTA',
         capex=0.0, opex=0.0, var_exp=2100, cov=0.02, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-014', name='Stress test FX+supply (12 combos)', dim='S1', prio='ALTA',
         capex=0.5, opex=0.0, var_exp=1850, cov=0.02, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-015', name='Recalibração semestral vol implícita', dim='S1', prio='MÉDIA',
         capex=0.0, opex=0.3, var_exp=2100, cov=0.01, p=0.30, cond=False, status='CONTÍNUO'),
    dict(ag='AG-016', name='Implementar tier system 0/1/2/3', dim='S4', prio='ALTA',
         capex=5.0, opex=33.5, var_exp=400, cov=0.29, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-018', name='Defensivo targeted Tier 2 (5k und)', dim='S4', prio='ALTA',
         capex=15.0, opex=0.0, var_exp=300, cov=0.25, p=0.25, cond=True, status='CONDICIONAL'),
    dict(ag='AG-019', name='Análise competitiva trimestral', dim='S4/S11', prio='ALTA',
         capex=0.0, opex=2.0, var_exp=300, cov=0.15, p=0.30, cond=False, status='CONTÍNUO'),
    dict(ag='AG-020', name='Game theory layer (Nash Differentiate)', dim='S4/S11', prio='ALTA',
         capex=1.5, opex=0.0, var_exp=500, cov=0.20, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-021', name='Qualificar EVE Tier 1 (12m)', dim='S2', prio='ALTA',
         capex=280.0, opex=0.0, var_exp=1480, cov=0.30, p=0.40, cond=False, status='PENDENTE'),
    dict(ag='AG-022', name='Contratos CATL LP 70% locked', dim='S2', prio='CRÍTICA',
         capex=340.0, opex=60.0, var_exp=1480, cov=0.85, p=0.45, cond=False, status='EM CURSO'),
    dict(ag='AG-023', name='Mapear 3 fornecedores alternativos', dim='S2', prio='MÉDIA',
         capex=0.5, opex=0.0, var_exp=400, cov=0.05, p=0.30, cond=False, status='PENDENTE'),
    dict(ag='AG-024', name='Safety stock +30 dias (S6 AMBER+)', dim='S2', prio='MÉDIA',
         capex=30.0, opex=0.0, var_exp=400, cov=0.15, p=0.35, cond=True, status='CONDICIONAL'),
    dict(ag='AG-025', name='Plano B spot purchasing (S6 RED)', dim='S2', prio='ALTA',
         capex=80.0, opex=0.0, var_exp=1480, cov=0.20, p=0.30, cond=True, status='CONDICIONAL'),
    dict(ag='AG-026', name='Auto-trigger S6 -> S1/S2/S3/S4/S5', dim='S6', prio='ALTA',
         capex=0.8, opex=0.0, var_exp=8210, cov=0.03, p=0.30, cond=False, status='PENDENTE'),
]

assert len(ACTIONS) == 25
assert len({a['ag'] for a in ACTIONS}) == 25
assert 'AG-017' not in {a['ag'] for a in ACTIONS}
print(f'25 ações | AG-017 REMOVIDA | críticas={sum(a["prio"]=="CRÍTICA" for a in ACTIONS)} | condicionais={sum(a["cond"] for a in ACTIONS)}')

25 ações | AG-017 REMOVIDA | críticas=11 | condicionais=4


## CELL 3 · Benefício esperado, NPV e ROI por ação

Fórmula: `benefit_anual = var_exp × cov × p_event` (perda anual evitada esperada). `npv = -capex + Σ (benefit - opex)/(1+r)^t`. `cost_pv = capex + Σ opex/(1+r)^t`. `roi = npv/cost_pv × 100`.

In [3]:
rows = []
for a in ACTIONS:
    benefit = a['var_exp'] * a['cov'] * a['p']
    cpv = cost_pv(a['capex'], a['opex'])
    action_npv = npv(a['capex'], a['opex'], benefit)
    roi = (action_npv / cpv * 100) if cpv > 0 else float('inf')
    rows.append({
        'ag': a['ag'], 'name': a['name'], 'dim': a['dim'], 'prio': a['prio'],
        'cond': a['cond'], 'status': a['status'],
        'var_exp': a['var_exp'], 'cov': a['cov'], 'p': a['p'],
        'benefit_anual': round(benefit, 2),
        'benefit_3y': round(benefit * H, 2),
        'capex': a['capex'], 'opex': a['opex'], 'cost_3y': round(a['capex'] + a['opex'] * H, 2),
        'cost_pv': round(cpv, 2),
        'npv': round(action_npv, 2),
        'roi_pct': round(roi, 1) if np.isfinite(roi) else None,
    })

df = pd.DataFrame(rows)
portfolio_npv = float(df['npv'].sum())
portfolio_npv_base = float(df.loc[~df['cond'], 'npv'].sum())
total_capex = float(df['capex'].sum())
total_cost3y = float(df['cost_3y'].sum())
print(f'NPV portfólio 25 ações: R$ {portfolio_npv:,.1f}M')
print(f'NPV base (sem cond.):   R$ {portfolio_npv_base:,.1f}M')
print(f'Capex total:            R$ {total_capex:,.1f}M')
print(f'Custo 3y total:         R$ {total_cost3y:,.1f}M')
print(f'NPV+: {int((df["npv"]>0).sum())}/25 | NPV-: {int((df["npv"]<=0).sum())}/25')
df.head(10)

NPV portfólio 25 ações: R$ 3,510.1M
NPV base (sem cond.):   R$ 3,307.2M
Capex total:            R$ 1,000.3M
Custo 3y total:         R$ 1,465.9M
NPV+: 24/25 | NPV-: 1/25


,ag,name,dim,prio,cond,status,var_exp,cov,p,benefit_anual,benefit_3y,capex,opex,cost_3y,cost_pv,npv,roi_pct
0,AG-001,Resolver lista suja S7 (kill switch),S7,CRÍTICA,False,EM CURSO,800,0.60,0.30,144.00,432.00,25.0,0.0,25.0,25.00,315.01,1260.0
1,AG-002,Implementar hedge FX 95%,S1,CRÍTICA,False,PENDENTE,2100,0.47,0.12,118.44,355.32,0.0,48.6,145.8,114.75,164.90,143.7
2,AG-003,Acelerar nacionalização 70%,S8,CRÍTICA,False,EM CURSO,650,0.85,0.40,221.00,663.00,150.0,10.0,180.0,173.61,348.20,200.6
3,AG-004,Diferenciação (tier + game theory),S4/S11,CRÍTICA,False,PENDENTE,500,0.40,0.30,60.00,180.00,6.5,0.0,6.5,6.50,135.17,2079.5
4,AG-005,Reserva lítio + safety stock,S2,CRÍTICA,False,EM CURSO,400,0.25,0.40,40.00,120.00,30.0,0.0,30.0,30.00,64.45,214.8
5,AG-006,Renegociar tariff quota Camex,S10,CRÍTICA,False,PENDENTE,2380,0.35,0.35,291.55,874.65,24.0,0.0,24.0,24.00,664.39,2768.3
6,AG-007,Definir resposta competitiva (Nash),S11,CRÍTICA,False,PENDENTE,300,0.30,0.30,27.00,81.00,6.0,0.0,6.0,6.00,57.75,962.5
7,AG-008,Plano contingência overcapacity,S11/S9,CRÍTICA,True,PENDENTE,200,0.30,0.20,12.00,36.00,4.0,0.0,4.0,4.00,24.33,608.3
8,AG-009,Setup compliance ESG + auditoria,S7,CRÍTICA,False,PLANEJADO,200,0.40,0.30,24.00,72.00,0.0,0.0,0.0,0.00,56.67,NaN
9,AG-010,Monitor T-MV1 (4-shock stress),S6,CRÍTICA,False,PENDENTE,12800,0.10,0.02,25.60,76.80,1.4,0.8,3.8,3.29,57.16,1737.9


## CELL 4 · Counterfactual — Com framework vs Sem framework

Os 6 eventos de stress backtested vêm do `LINHAGEM.md` Cap 11 (COVID 2020, Semicondutor 2021, Eleição 2022, Spike lítio 2022, Eleição 2024, Stagflation 2025) — totalizam **R$ 200 M+/ano** de perda evitada quando o framework está ativo. Sem framework, a exposição agregada é **VaR 4-shock R$ 8,21 bi / CVaR R$ 10,14 bi**, e a resposta a sinais é manual (4-8h por signal, decisão em semanas).

In [4]:
stress_events = [
    {'event': 'COVID 2020',        'year': 2020, 'avoided_rsm': 50},
    {'event': 'Semicondutor 2021', 'year': 2021, 'avoided_rsm': 30},
    {'event': 'Eleição 2022',      'year': 2022, 'avoided_rsm': 20},
    {'event': 'Spike lítio 2022',  'year': 2022, 'avoided_rsm': 80},
    {'event': 'Eleição 2024',      'year': 2024, 'avoided_rsm': 15},
    {'event': 'Stagflation 2025',  'year': 2025, 'avoided_rsm':  5},
]
stress_avoided_total = sum(e['avoided_rsm'] for e in stress_events)

counterfactual = {
    'com_framework': {
        'framework_cost_rsm': 3.0,
        'stress_avoided_rsm_ano': stress_avoided_total,
        'deteccao_dias': 14.0, 'resposta_dias': 9.3,
        'var_residual_note': 'VaR mitigado via hedge/dual-sourcing/triggers',
    },
    'sem_framework': {
        'framework_cost_rsm': 0.0,
        'stress_avoided_rsm_ano': 0.0,
        'resposta_horas_por_signal': '4-8h manual, decisão em semanas',
        'var_4shock_rbi': 8.21, 'cvar_95_rbi': 10.14,
    },
    'delta_ano_rsm': stress_avoided_total,
}

for e in stress_events:
    print(f"  {e['event']:<22} evitado R$ {e['avoided_rsm']:>3}M")
print(f"  {'TOTAL/ano':<22} evitado R$ {stress_avoided_total:>3}M+")
print('  Sem framework: VaR 4-shock R$ 8,21 bi | CVaR R$ 10,14 bi | resposta 4-8h/signal')

  COVID 2020             evitado R$  50M
  Semicondutor 2021      evitado R$  30M
  Eleição 2022           evitado R$  20M
  Spike lítio 2022       evitado R$  80M
  Eleição 2024           evitado R$  15M
  Stagflation 2025       evitado R$   5M
  TOTAL/ano              evitado R$ 200M+
  Sem framework: VaR 4-shock R$ 8,21 bi | CVaR R$ 10,14 bi | resposta 4-8h/signal


## CELL 5 · ROI headline — R$ 3M investido, R$ 200M+ evitado/ano, ROI 200×

O **framework D3 sozinho** (camadas F1-F3, 28 semanas de construção) custa R$ 3M e evita R$ 200M+/ano de stress. Aritmética: R$ 600M evitados em 3 anos ÷ R$ 3M investidos = **200× ROI (3 anos)**, **66,7× anual**. Payback = R$ 3M / (R$ 200M/12) = **0,18 mês** (< 1 mês).

In [5]:
framework_cost = 3.0
avoided_per_year = float(stress_avoided_total)
avoided_3y = avoided_per_year * H
roi_3y_x = avoided_3y / framework_cost
roi_anual_x = avoided_per_year / framework_cost
payback_meses = framework_cost / (avoided_per_year / 12.0)
framework_npv = npv(framework_cost, 0.0, avoided_per_year)

print(f'Investimento framework: R$ {framework_cost:.0f}M (F1-F3, 28 semanas)')
print(f'Stress evitado:         R$ {avoided_per_year:.0f}M+/ano -> R$ {avoided_3y:.0f}M em 3 anos')
print(f'ROI (3 anos): {roi_3y_x:.0f}x | ROI anual: {roi_anual_x:.1f}x')
print(f'Payback: {payback_meses:.2f} meses (< 1 mês)')
print(f'NPV framework (3y @ 13%): R$ {framework_npv:,.1f}M')
assert round(roi_3y_x) == 200

Investimento framework: R$ 3M (F1-F3, 28 semanas)
Stress evitado:         R$ 200M+/ano -> R$ 600M em 3 anos
ROI (3 anos): 200x | ROI anual: 66.7x
Payback: 0.18 meses (< 1 mês)
NPV framework (3y @ 13%): R$ 469.2M


## CELL 6 · NPV Tornado — sensibilidade aos drivers globais

O tornado estressa o NPV total do portfólio (25 ações) variando cada driver global em ±X%. Os 3 maiores swings: probabilidade de shock (p), exposição VaR e cobertura esperada — confirma que **o valor do portfólio é dominado pela modelagem probabilística dos eventos, não pela taxa de desconto ou pelos custos**.

In [6]:
def portfolio_npv_with(r=R, p_mult=1.0, cov_mult=1.0, var_mult=1.0, opex_mult=1.0, capex_mult=1.0):
    total = 0.0
    for a in ACTIONS:
        benefit = (a['var_exp']*var_mult) * min(a['cov']*cov_mult, 1.0) * min(a['p']*p_mult, 1.0)
        total += npv(a['capex']*capex_mult, a['opex']*opex_mult, benefit, r=r)
    return total

base_np = portfolio_npv_with()
drivers = [
    ('Taxa de desconto r (±3pp)',   dict(r=R+0.03),         dict(r=R-0.03)),
    ('Prob. shock p (±30%)',        dict(p_mult=0.70),      dict(p_mult=1.30)),
    ('Cobertura cov (±20%)',        dict(cov_mult=0.80),    dict(cov_mult=1.20)),
    ('Exposição VaR (±25%)',        dict(var_mult=0.75),    dict(var_mult=1.25)),
    ('Opex recorrente (±30%)',      dict(opex_mult=1.30),   dict(opex_mult=0.70)),
    ('Capex (±20%)',                dict(capex_mult=1.20),  dict(capex_mult=0.80)),
]
tornado = []
for label, low_kw, high_kw in drivers:
    lo = portfolio_npv_with(**low_kw)
    hi = portfolio_npv_with(**high_kw)
    tornado.append({'driver': label, 'low': lo, 'high': hi, 'swing': abs(hi - lo)})
tornado = sorted(tornado, key=lambda d: d['swing'], reverse=True)

print(f'NPV base portfólio: R$ {base_np:,.1f}M')
for t in tornado:
    print(f"  {t['driver']:<28} low=R${t['low']:>8,.0f}M  high=R${t['high']:>8,.0f}M  swing=R${t['swing']:>7,.0f}M")

# Matplotlib PNG
fig, ax = plt.subplots(figsize=(10, 5.5))
ypos = np.arange(len(tornado))
for i, t in enumerate(tornado):
    lo, hi = sorted([t['low'], t['high']])
    ax.barh(i, hi - lo, left=lo, color='#3b82f6', edgecolor='#1e3a8a', height=0.6)
ax.axvline(base_np, color='#ef4444', ls='--', lw=1.5, label=f'NPV base R${base_np:,.0f}M')
ax.set_yticks(ypos); ax.set_yticklabels([t['driver'] for t in tornado])
ax.invert_yaxis()
ax.set_xlabel('NPV do portfólio (R$ M)')
ax.set_title('NB-14 · NPV Tornado — drivers globais', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'nb14_tornado.png'), dpi=130)
plt.close(fig)

# Plotly HTML
figp = go.Figure()
for t in tornado:
    lo, hi = sorted([t['low'], t['high']])
    figp.add_trace(go.Bar(y=[t['driver']], x=[hi-lo], base=lo, orientation='h',
                          marker_color='#3b82f6', showlegend=False,
                          hovertemplate=f"{t['driver']}<br>low R${t['low']:,.0f}M<br>high R${t['high']:,.0f}M<extra></extra>"))
figp.add_vline(x=base_np, line_dash='dash', line_color='#ef4444',
               annotation_text=f'base R${base_np:,.0f}M')
figp.update_layout(title='NB-14 · NPV Tornado — drivers globais',
                   xaxis_title='NPV portfólio (R$ M)', template='plotly_white',
                   height=430, margin=dict(l=220, r=40, t=60, b=40))
figp.write_html(os.path.join(OUTPUT_DIR, 'nb14_tornado.html'), include_plotlyjs='cdn')
print('Tornado salvo: nb14_tornado.png + .html')

NPV base portfólio: R$ 3,510.0M
  Prob. shock p (±30%)         low=R$   2,047M  high=R$   4,973M  swing=R$  2,926M
  Exposição VaR (±25%)         low=R$   2,291M  high=R$   4,729M  swing=R$  2,438M
  Cobertura cov (±20%)         low=R$   2,535M  high=R$   4,442M  swing=R$  1,907M
  Taxa de desconto r (±3pp)    low=R$   3,290M  high=R$   3,750M  swing=R$    460M
  Capex (±20%)                 low=R$   3,310M  high=R$   3,710M  swing=R$    400M
  Opex recorrente (±30%)       low=R$   3,400M  high=R$   3,620M  swing=R$    220M


Tornado salvo: nb14_tornado.png + .html


## CELL 7 · Ranking por NPV

Top-1 é **AG-022 Contratos CATL LP 70% locked** com NPV R$ 855M (capex R$ 340M, opex R$ 60M/ano × 3y, exposição R$ 1.480M, cobertura 85%, probabilidade anual 45%). Top-2 é **AG-006 Renegociar tariff quota Camex** com NPV R$ 664M e ROI 2.768% — a ação mais barata e de maior retorno. Apenas **AG-016 tier system** fica marginalmente negativa (NPV -R$ 2M), porque o opex recorrente (R$ 33,5M/ano) supera o benefício esperado em uma base ampla.

In [7]:
ranked = df.sort_values('npv', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)
display(ranked[['rank', 'ag', 'name', 'dim', 'npv', 'roi_pct', 'cost_3y']])

# Charts
colors = ['#16a34a' if v > 0 else '#dc2626' for v in ranked['npv']]
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(ranked['ag'] + ' ' + ranked['name'].str.slice(0, 26), ranked['npv'], color=colors)
ax.invert_yaxis()
ax.axvline(0, color='#334155', lw=1)
ax.set_xlabel('NPV (R$ M)')
ax.set_title('NB-14 · Ranking das 25 ações por NPV (3y @ 13%)', fontweight='bold')
ax.grid(axis='x', alpha=0.3); ax.tick_params(axis='y', labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'nb14_ranking.png'), dpi=130)
plt.close(fig)

figp2 = go.Figure(go.Bar(
    x=ranked['npv'], y=ranked['ag'] + ' ' + ranked['name'].str.slice(0, 30),
    orientation='h', marker_color=colors,
    hovertemplate='%{y}<br>NPV R$%{x:.1f}M<extra></extra>'))
figp2.update_layout(title='NB-14 · Ranking das 25 ações por NPV', xaxis_title='NPV (R$ M)',
                    template='plotly_white', height=720, yaxis=dict(autorange='reversed'),
                    margin=dict(l=280, r=40, t=60, b=40))
figp2.write_html(os.path.join(OUTPUT_DIR, 'nb14_ranking.html'), include_plotlyjs='cdn')
print('Ranking salvo: nb14_ranking.png + .html')

,rank,ag,name,dim,npv,roi_pct,cost_3y
0,1,AG-022,Contratos CATL LP 70% locked,S2,854.98,177.5,520.00
1,2,AG-006,Renegociar tariff quota Camex,S10,664.39,2768.3,24.00
2,3,AG-003,Acelerar nacionalização 70%,S8,348.20,200.6,180.00
3,4,AG-001,Resolver lista suja S7 (kill switch),S7,315.01,1260.0,25.00
4,5,AG-026,Auto-trigger S6 -> S1/S2/S3/S4/S5,S6,173.67,21708.2,0.80
5,6,AG-002,Implementar hedge FX 95%,S1,164.90,143.7,145.80
6,7,AG-021,Qualificar EVE Tier 1 (12m),S2,139.34,49.8,280.00
7,8,AG-004,Diferenciação (tier + game theory),S4/S11,135.17,2079.5,6.50
8,9,AG-025,Plano B spot purchasing (S6 RED),S2,129.67,162.1,80.00
9,10,AG-020,Game theory layer (Nash Differentiate),S4/S11,69.33,4622.3,1.50


Ranking salvo: nb14_ranking.png + .html


## CELL 8 · Decision Matrix

Regras de decisão:
- `EXECUTAR JÁ (90d)` — NPV>0 + prioridade CRÍTICA
- `EXECUTAR` — NPV>0 + prioridade ALTA/MÉDIA
- `CONDICIONAL (trigger)` — NPV>0 mas flag `cond=True` (só dispara se gatilho S6 acionado)
- `MONITORAR / REAVALIAR` — NPV≤0
- **REJEITAR (AG-017)** — catalog-wide REMOVIDA por ROI -67% (contraprova empírica)

**Resumo:** 10 EXECUTAR JÁ + 10 EXECUTAR + 4 CONDICIONAL + 1 MONITORAR + 1 REJEITAR = 25 ações + 1 contraprova.

In [8]:
def decide(r):
    if r['npv'] <= 0:
        return 'MONITORAR / REAVALIAR'
    if r['cond']:
        return 'CONDICIONAL (trigger)'
    if r['prio'] == 'CRÍTICA':
        return 'EXECUTAR JÁ (90d)'
    return 'EXECUTAR'

dm = ranked.copy()
dm['decisao'] = dm.apply(decide, axis=1)
display(dm[['rank', 'ag', 'name', 'dim', 'prio', 'npv', 'roi_pct', 'decisao']])
print('\nResumo decisões:')
for k, v in dm['decisao'].value_counts().items():
    print(f'  {k:<26} {v}')
print('  + REJEITAR: AG-017 catalog-wide (ROI -67%, contraprova)')

,rank,ag,name,dim,prio,npv,roi_pct,decisao
0,1,AG-022,Contratos CATL LP 70% locked,S2,CRÍTICA,854.98,177.5,EXECUTAR JÁ (90d)
1,2,AG-006,Renegociar tariff quota Camex,S10,CRÍTICA,664.39,2768.3,EXECUTAR JÁ (90d)
2,3,AG-003,Acelerar nacionalização 70%,S8,CRÍTICA,348.20,200.6,EXECUTAR JÁ (90d)
3,4,AG-001,Resolver lista suja S7 (kill switch),S7,CRÍTICA,315.01,1260.0,EXECUTAR JÁ (90d)
4,5,AG-026,Auto-trigger S6 -> S1/S2/S3/S4/S5,S6,ALTA,173.67,21708.2,EXECUTAR
5,6,AG-002,Implementar hedge FX 95%,S1,CRÍTICA,164.90,143.7,EXECUTAR JÁ (90d)
6,7,AG-021,Qualificar EVE Tier 1 (12m),S2,ALTA,139.34,49.8,EXECUTAR
7,8,AG-004,Diferenciação (tier + game theory),S4/S11,CRÍTICA,135.17,2079.5,EXECUTAR JÁ (90d)
8,9,AG-025,Plano B spot purchasing (S6 RED),S2,ALTA,129.67,162.1,CONDICIONAL (trigger)
9,10,AG-020,Game theory layer (Nash Differentiate),S4/S11,ALTA,69.33,4622.3,EXECUTAR



Resumo decisões:
  EXECUTAR JÁ (90d)          10
  EXECUTAR                   10
  CONDICIONAL (trigger)      4
  MONITORAR / REAVALIAR      1
  + REJEITAR: AG-017 catalog-wide (ROI -67%, contraprova)


## CELL 9 · Export JSON

Exporta `outputs/nb14_results.json` com 25 ações + portfolio + counterfactual + stress events + ROI headline + tornado + ranking + decision matrix. Este JSON é a fonte canônica para qualquer agregação downstream (dashboards, 1-pagers, anexos do D3).

In [9]:
results = {
    'notebook': 'NB-14 NPV Layer — Cost-Benefit das 25 prescrições',
    'computed_at': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'params': {'r': R, 'horizon': H, 'annuity_factor': round(ANNUITY, 6),
               'discount_note': 'NTN-B 10y (~11%) + 200bps spread soberano = 13% nominal'},
    'n_actions': len(ACTIONS),
    'actions': df.to_dict(orient='records'),
    'portfolio': {
        'npv_total_rsm': round(portfolio_npv, 1),
        'npv_base_rsm': round(portfolio_npv_base, 1),
        'capex_total_rsm': round(total_capex, 1),
        'cost_3y_total_rsm': round(total_cost3y, 1),
        'n_npv_positive': int((df['npv'] > 0).sum()),
        'n_npv_negative': int((df['npv'] <= 0).sum()),
    },
    'counterfactual': counterfactual,
    'stress_events': stress_events,
    'roi_headline': {
        'framework_cost_rsm': framework_cost,
        'avoided_per_year_rsm': avoided_per_year,
        'avoided_3y_rsm': avoided_3y,
        'roi_3y_x': round(roi_3y_x, 1),
        'roi_anual_x': round(roi_anual_x, 1),
        'payback_meses': round(payback_meses, 2),
        'framework_npv_rsm': round(framework_npv, 1),
    },
    'tornado': {'base_npv_rsm': round(base_np, 1), 'drivers': [
        {'driver': t['driver'], 'low': round(t['low'], 1), 'high': round(t['high'], 1),
         'swing': round(t['swing'], 1)} for t in tornado]},
    'ranking': ranked[['rank', 'ag', 'name', 'dim', 'npv', 'roi_pct', 'cost_3y']].to_dict(orient='records'),
    'decision_matrix': dm[['rank', 'ag', 'name', 'dim', 'prio', 'npv', 'roi_pct', 'decisao']].to_dict(orient='records'),
    'decision_summary': dm['decisao'].value_counts().to_dict(),
    'outputs': {
        'tornado_png': 'nb14_tornado.png', 'tornado_html': 'nb14_tornado.html',
        'ranking_png': 'nb14_ranking.png', 'ranking_html': 'nb14_ranking.html',
    },
}
out_path = os.path.join(OUTPUT_DIR, 'nb14_results.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('Saved:', out_path)

Saved: C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva\outputs\nb14_results.json


---

## Resumo final

| Métrica | Valor |
|---|---|
| Ações analisadas | **25** (AG-001..AG-026, AG-017 REMOVIDA) |
| NPV portfólio total | **R$ 3.510 M** (3y @ 13%) |
| NPV base (sem condicionais) | R$ 3.307 M |
| Capex total | R$ 1.000 M |
| Custo 3y total | R$ 1.466 M |
| Ações com NPV+ | 24/25 |
| Top-1 NPV | AG-022 CATL LP 70% locked (R$ 855 M) |
| Top-1 ROI | AG-006 Tariff quota Camex (2.768%) |
| **Framework D3** | **R$ 3M → R$ 200M+/ano evitado → ROI 200× (3y)** |
| Payback | **0,18 mês** (< 1 mês) |
| Sem framework | VaR 4-shock R$ 8,21 bi / CVaR R$ 10,14 bi |
| Decisões | 10 EXECUTAR JÁ + 10 EXECUTAR + 4 CONDICIONAL + 1 MONITORAR + 1 REJEITAR |

**Outputs gerados em `outputs/`:**
- `nb14_results.json` — fonte canônica de dados
- `nb14_tornado.png` + `.html` — sensibilidade NPV aos drivers globais
- `nb14_ranking.png` + `.html` — ranking visual das 25 ações por NPV